# Probes, Frequencies, and Ablations

Read representations, then intervene before claiming a mechanism.

## Before You Start

Course 11. Entry course: 1. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** dynamics-03_modular.

## What You Will Be Able to Do

- Fit a frozen linear probe
- Decompose activation grids into frequencies
- Compare restricted and excluded predictions

## Readable information is not necessarily used information

A linear probe maps fixed hidden features H [N,D] to labels or targets. Freeze H and fit only the probe on a training subset, then evaluate on held-out rows. A high probe score shows decodability under that probe and split. It does not prove the original model used the decoded information, nor that a neuron has a unique semantic meaning. Label shuffling and simple baselines help expose overfitting.

For a modular-addition model, record one neuron over all a,b pairs as a p-by-p grid. A diagonal or periodic pattern suggests a hypothesis. Apply a two-dimensional discrete Fourier transform to quantify frequency components instead of relying only on color. A real grid has conjugate-symmetric coefficients; keep conjugate pairs if an inverse transform must remain real.

## Restricted and excluded components

Given a declared frequency mask, reconstruct selected components by zeroing all other Fourier coefficients. The excluded reconstruction retains the complement. Their sum must recover the original grid. In this lab the grid can be an activation or a logit slice. Restricted loss uses predictions built only from selected components; excluded loss uses the complement. Compute loss against the same held-out targets and document which tensor was altered.

In the grokking case study, circuit-specific progress measures exposed changes not obvious from accuracy alone. Our generic Fourier filter teaches the intervention mechanics, not an automatic discovery of that circuit in any model. A low restricted loss with high excluded loss supports the importance of the chosen component under the intervention; distribution shift and nonlinear downstream compensation still limit causal claims.

## Worked Example: A reconstructable intervention

Grid = constant component + oscillating component.

1. **Select DC:** The [0,0] Fourier coefficient captures the grid mean.
2. **Restrict:** Keeping only DC reconstructs a constant grid at that mean.
3. **Exclude:** Removing DC reconstructs the zero-mean oscillation. Restricted plus excluded equals the original.

State exactly what you remove, and test reconstruction before interpreting a loss change.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Frozen linear probes and explicit Fourier interventions."""
import numpy as np

def fit_probe(features, targets, ridge=0.0):
    """Fit W [D,K] to finite features [N,D], targets [N,K], no intercept.

Minimize ||HW-Y||^2 + ridge*||W||^2. ridge>=0; otherwise ValueError.
Return the minimum-norm solution when ridge=0. No feature updates."""
    raise NotImplementedError('Implement this function, then run the checks.')

def fourier_parts(grid, keep):
    """Return (restricted, excluded) complex inverse-FFT grids.

grid finite 2D, keep matching Boolean mask. Complex results are intentional:
nonsymmetric masks need not reconstruct real signals. Their sum must equal
grid. Do not discard imaginary parts or modify inputs."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

A grid is [[1,3],[5,7]]. Keep only its DC frequency.

- Every reconstructed cell

<details><summary>Worked answers</summary>

- 4: The DC-only reconstruction is the mean: 16/4=4.

</details>

## Experiment

Run `experiments/dynamics_modular.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- Measure a held-out probe on frozen features, and compare with shuffled-label training.
- Fourier energy is descriptive; the experiment prints reconstruction error and component-loss changes separately.

## Transfer Problems

1. Test a probe on random features with more dimensions than training examples.
2. Design an intervention that distinguishes correlation from actual use of a feature.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[Progress Measures for Grokking](https://arxiv.org/abs/2301.05217). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.



## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Frozen linear probes and explicit Fourier interventions."""
import numpy as np

def fit_probe(features, targets, ridge=0.0):
    """Fit W [D,K] to finite features [N,D], targets [N,K], no intercept.

Minimize ||HW-Y||^2 + ridge*||W||^2. ridge>=0; otherwise ValueError.
Return the minimum-norm solution when ridge=0. No feature updates."""
    raise NotImplementedError('Implement this function, then run the checks.')

def fourier_parts(grid, keep):
    """Return (restricted, excluded) complex inverse-FFT grids.

grid finite 2D, keep matching Boolean mask. Complex results are intentional:
nonsymmetric masks need not reconstruct real signals. Their sum must equal
grid. Do not discard imaginary parts or modify inputs."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np
h = np.array([[1., 0.], [0., 1.], [1., 1.]])
y = np.array([[2.], [3.], [5.]])
assert np.allclose(fit_probe(h, y), [[2], [3]])
got = fit_probe(h, y, .4)
assert np.allclose((h.T@h+.4*np.eye(2)) @ got, h.T@y)
grid = np.array([[1., 3.], [5., 7.]])
mask = np.array([[1, 0], [0, 0]], bool)
restricted, excluded = fourier_parts(grid, mask)
assert np.allclose(restricted, 4) and np.allclose(excluded, grid-4)
rng = np.random.default_rng(6)
grid = rng.normal(size=(5, 5)); mask = rng.random((5, 5)) > .6
a, b = fourier_parts(grid, mask)
assert np.allclose(a+b, grid)
assert np.allclose(np.fft.fft2(a)[~mask], 0, atol=1e-10)
print('Probe and Fourier checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Frozen linear probes and explicit Fourier interventions."""
import numpy as np


def fit_probe(features, targets, ridge=0.):
    """Fit W [D,K] to finite features [N,D], targets [N,K], no intercept.

    Minimize ||HW-Y||^2 + ridge*||W||^2. ridge>=0; otherwise ValueError.
    Return the minimum-norm solution when ridge=0. No feature updates.
    """
    if not np.isfinite(ridge) or ridge < 0:
        raise ValueError('invalid ridge')
    h, y = np.asarray(features, float), np.asarray(targets, float)
    design = np.vstack([h, np.sqrt(ridge)*np.eye(h.shape[1])])
    response = np.vstack([y, np.zeros((h.shape[1], y.shape[1]))])
    return np.linalg.lstsq(design, response, rcond=None)[0]


def fourier_parts(grid, keep):
    """Return (restricted, excluded) complex inverse-FFT grids.

    grid finite 2D, keep matching Boolean mask. Complex results are intentional:
    nonsymmetric masks need not reconstruct real signals. Their sum must equal
    grid. Do not discard imaginary parts or modify inputs.
    """
    spectrum = np.fft.fft2(grid)
    return np.fft.ifft2(np.where(keep, spectrum, 0)), np.fft.ifft2(np.where(keep, 0, spectrum))


In [ ]:
import numpy as np
h = np.array([[1., 0.], [0., 1.], [1., 1.]])
y = np.array([[2.], [3.], [5.]])
assert np.allclose(fit_probe(h, y), [[2], [3]])
got = fit_probe(h, y, .4)
assert np.allclose((h.T@h+.4*np.eye(2)) @ got, h.T@y)
grid = np.array([[1., 3.], [5., 7.]])
mask = np.array([[1, 0], [0, 0]], bool)
restricted, excluded = fourier_parts(grid, mask)
assert np.allclose(restricted, 4) and np.allclose(excluded, grid-4)
rng = np.random.default_rng(6)
grid = rng.normal(size=(5, 5)); mask = rng.random((5, 5)) > .6
a, b = fourier_parts(grid, mask)
assert np.allclose(a+b, grid)
assert np.allclose(np.fft.fft2(a)[~mask], 0, atol=1e-10)
print('Probe and Fourier checks passed.')


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""Actual modular training, frozen probes, and frequency interventions.

Default 300 updates is a smoke run, NOT a promised grokking reproduction.
To investigate longer: MODULAR_STEPS=20000 python experiments/dynamics_modular.py
Reports all seeds; no external data. This MLP differs from the paper Transformer.
"""
import json
import os
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F


def modular_experiment(steps=300):
    torch.set_num_threads(1)
    modulus = 17
    pairs = torch.cartesian_prod(torch.arange(modulus), torch.arange(modulus))
    inputs = torch.cat([F.one_hot(pairs[:, 0], modulus), F.one_hot(pairs[:, 1], modulus)], -1).float()
    targets = pairs.sum(-1) % modulus
    split = torch.randperm(len(pairs), generator=torch.Generator().manual_seed(44))
    train, test = split[:100], split[100:]
    reports = []
    for seed in [3, 8, 21]:
        torch.manual_seed(seed)
        hidden = nn.Sequential(nn.Linear(2*modulus, 64), nn.ReLU())
        head = nn.Linear(64, modulus)
        model = nn.Sequential(hidden, head)
        optimizer = torch.optim.AdamW(model.parameters(), lr=.003, weight_decay=.1)
        curve = []
        for step in range(steps):
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(inputs[train]), targets[train]); loss.backward(); optimizer.step()
            if step % max(1, steps//10) == 0 or step == steps-1:
                with torch.no_grad():
                    logits = model(inputs)
                    curve.append(dict(step=step+1, train_accuracy=float((logits[train].argmax(-1)==targets[train]).float().mean()), test_accuracy=float((logits[test].argmax(-1)==targets[test]).float().mean())))
        with torch.no_grad():
            features = hidden(inputs).numpy()
            logits = model(inputs).numpy().astype(np.float64).reshape(modulus, modulus, modulus)
        onehot = np.eye(modulus)[targets.numpy()]
        h = features[train]; y = onehot[train]
        weights = np.linalg.solve(h.T@h + np.eye(64), h.T@y)
        probe_accuracy = float((features[test]@weights).argmax(-1).__eq__(targets[test].numpy()).mean())
        shuffled = np.random.default_rng(seed).permutation(y)
        shuffled_weights = np.linalg.solve(h.T@h + np.eye(64), h.T@shuffled)
        shuffled_accuracy = float(((features[test]@shuffled_weights).argmax(-1)==targets[test].numpy()).mean())
        spectrum = np.fft.fft2(logits, axes=(0, 1))
        energy = (np.abs(spectrum)**2).sum(-1)
        # Predeclared illustrative diagonal frequencies, not a discovered circuit.
        mask = np.eye(modulus, dtype=bool)
        restricted = np.fft.ifft2(np.where(mask[..., None], spectrum, 0), axes=(0, 1)).real
        excluded = logits - restricted
        def heldout_loss(values):
            return float(F.cross_entropy(torch.tensor(values.reshape(-1, modulus)[test]), targets[test]))
        assert np.allclose(restricted+excluded, logits)
        reports.append(dict(seed=seed, steps=steps, train_pairs=len(train), test_pairs=len(test), curve=curve,
            probe_accuracy=probe_accuracy, shuffled_probe_accuracy=shuffled_accuracy,
            total_loss=heldout_loss(logits), restricted_loss=heldout_loss(restricted), excluded_loss=heldout_loss(excluded),
            diagonal_frequency_energy_fraction=float(energy[mask].sum()/energy.sum()),
            neuron0_grid=features[:, 0].reshape(modulus, modulus).round(3).tolist(),
            delayed_transition_observed=any(p['train_accuracy']>.95 and p['test_accuracy']<.3 for p in curve[:-1]) and curve[-1]['test_accuracy']>.9))
    return reports


if __name__ == '__main__':
    print(json.dumps(modular_experiment(int(os.environ.get('MODULAR_STEPS', '300'))), indent=2))
